# Price Prediction Model — Archive Fashion Items

This notebook builds a price prediction model for the top 5 most-traded archive fashion items on Grailed. We compare **Linear Regression** (baseline) vs **XGBoost** using time-series features and rolling price averages.

**Pipeline:**
1. Data loading & outlier removal
2. Feature engineering (time, condition, rolling averages)
3. Model training (80/20 time-series split)
4. Evaluation (MAE / RMSE comparison)
5. Visualization (actual vs predicted)
6. Summary table with price trend signals

---

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import Markdown, display

import price_model

print("Modules loaded.")

## 1. Load & Clean Data

We load `historical_sold.csv`, filter to items with ≥30 records, and remove price outliers that deviate more than 3x from the median. This prevents extreme listings (mispriced or bundled items) from skewing the model.

In [ ]:
df_raw = price_model.load_data()
print(f"Raw data: {len(df_raw)} records, {df_raw['keyword'].nunique()} items")
print(f"Date range: {df_raw['sold_date'].min().strftime('%Y-%m-%d')} ~ {df_raw['sold_date'].max().strftime('%Y-%m-%d')}")
print()

# Per-item counts before cleaning
print("Records per item:")
for kw, g in df_raw.groupby("keyword"):
    print(f"  {kw:<35} {len(g):>3} records, median ${g['sold_price'].median():.0f}")

print("\nRemoving outliers (>3x median)...")
df_clean = price_model.remove_outliers(df_raw)
print(f"\nAfter cleanup: {len(df_clean)} records ({len(df_raw) - len(df_clean)} removed)")

## 2. Feature Engineering

We extract features from each transaction:

| Feature | Source | Rationale |
|---------|--------|-----------|
| `week_of_year` | sold_date | Captures seasonal demand cycles |
| `month` | sold_date | Monthly trends (holiday spikes, etc.) |
| `day_of_week` | sold_date | Weekend vs weekday buying patterns |
| `days_since_start` | sold_date | Linear time trend |
| `condition_code` | condition | New (4) → Worn (1); better condition = higher price |
| `followers` | followers | Listing hype / demand signal |
| `rolling_avg_7/14/30d` | sold_price history | Recent price momentum — the most predictive features |

In [ ]:
print("Building features (rolling averages take a moment)...")
df = price_model.build_features(df_clean)

# Show feature sample
display(df[["keyword", "sold_date", "sold_price", "condition_code", "followers",
            "week_of_year", "month", "day_of_week", "days_since_start",
            "rolling_avg_7d", "rolling_avg_14d", "rolling_avg_30d"]].head(10))

print(f"\nFeature matrix shape: {df[price_model.FEATURE_COLS].shape}")
print(f"Features: {price_model.FEATURE_COLS}")

## 3. Model Training & Evaluation

We use an **80/20 time-series split** (not random split — this respects temporal ordering and prevents future data leakage). Two models are compared:

- **Linear Regression**: Simple baseline. Assumes linear relationship between features and price.
- **XGBoost**: Gradient-boosted trees. Captures non-linear patterns and feature interactions.

Metrics:
- **MAE** (Mean Absolute Error): Average dollar error — interpretable as "the model is off by $X on average"
- **RMSE** (Root Mean Squared Error): Penalizes large errors more heavily

In [ ]:
print("Training models per item...\n")
results = price_model.train_and_evaluate(df)

## 4. Visualization — Actual vs Predicted

Each chart shows:
- **Blue dots**: Actual sold prices
- **Red line**: XGBoost predicted trend (or Linear Regression if XGBoost unavailable)
- **Gray dashed line**: Linear Regression baseline
- **Green band**: Train/test split point

In [ ]:
for r in results:
    kw = r["keyword"]
    dates = pd.to_datetime(r["dates"])
    actuals = r["actuals"]
    lr_pred = r["lr_pred_all"]
    split_idx = r["train_size"]
    split_date = dates[split_idx]

    fig = go.Figure()

    # Actual prices (blue dots)
    fig.add_trace(go.Scatter(
        x=dates, y=actuals, mode="markers",
        name="Actual Sold Price",
        marker=dict(color="#2563eb", size=7, opacity=0.7),
    ))

    # LR baseline (gray dashed)
    fig.add_trace(go.Scatter(
        x=dates, y=lr_pred, mode="lines",
        name=f"Linear Regression (MAE=${r['lr_mae']:.0f})",
        line=dict(color="#9ca3af", width=2, dash="dash"),
    ))

    # XGBoost (red solid)
    if "xgb_pred_all" in r:
        fig.add_trace(go.Scatter(
            x=dates, y=r["xgb_pred_all"], mode="lines",
            name=f"XGBoost (MAE=${r['xgb_mae']:.0f})",
            line=dict(color="#dc2626", width=2),
        ))

    # Train/test split line
    fig.add_vline(x=split_date, line_dash="dot", line_color="#059669",
                  annotation_text="Train | Test", annotation_position="top right")

    trend_emoji = {"Rising": "↑", "Declining": "↓", "Stable": "→"}[r["trend"]]

    fig.update_layout(
        title=f"{kw}  —  Predicted: ${r['predicted_price']:.0f}  {trend_emoji} {r['trend']} ({r['pct_change']:+.1f}%)",
        xaxis_title="Date", yaxis_title="Price (USD)",
        height=380,
        margin=dict(l=0, r=20, t=50, b=20),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    )
    fig.show()

## 5. Model Comparison — MAE Bar Chart

Side-by-side comparison of Linear Regression vs XGBoost MAE per item. Lower is better.

In [ ]:
items = [r["keyword"] for r in results]
lr_maes = [r["lr_mae"] for r in results]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=items, y=lr_maes, name="Linear Regression",
    marker_color="#9ca3af", text=[f"${v:.0f}" for v in lr_maes], textposition="outside",
))

if "xgb_mae" in results[0]:
    xgb_maes = [r["xgb_mae"] for r in results]
    fig.add_trace(go.Bar(
        x=items, y=xgb_maes, name="XGBoost",
        marker_color="#dc2626", text=[f"${v:.0f}" for v in xgb_maes], textposition="outside",
    ))

fig.update_layout(
    title="Model Comparison — MAE per Item (lower is better)",
    yaxis_title="MAE ($)",
    barmode="group",
    height=400,
    margin=dict(l=0, r=20, t=40, b=20),
)
fig.show()

## 6. Summary Table

Final output: predicted price, 30-day average, and trend direction for each item.

- **Rising (↑):** Predicted price > 30-day avg by more than 5%
- **Declining (↓):** Predicted price < 30-day avg by more than 5%
- **Stable (→):** Within ±5% of 30-day average

In [ ]:
summary = price_model.build_summary_table(results)
display(summary)

# Markdown interpretation
display(Markdown("---\n### Interpretation\n"))
for r in results:
    emoji = {"Rising": "📈", "Declining": "📉", "Stable": "➡️"}[r["trend"]]
    display(Markdown(
        f"- **{r['keyword']}** {emoji} {r['trend']} ({r['pct_change']:+.1f}%) — "
        f"Predicted **${r['predicted_price']:.0f}** vs 30-day avg ${r['avg_30d']:.0f}  "
        f"(Best model MAE: ${min(r['lr_mae'], r.get('xgb_mae', r['lr_mae'])):.0f})"
    ))